# DCGAN vs. Convolutional Beta-VAE on MNIST

This notebook compares adversarial and variational image generation on MNIST. It uses fixed random seeds, a shared held-out reference set, and the same sample count for both feature-space evaluations.

The InceptionV3-based score is used only for comparison inside this experiment. ImageNet features are not domain-specific to grayscale handwritten digits.

## 1. Environment and experiment configuration

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from scipy.linalg import sqrtm
from tensorflow.keras import Model, layers
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

SEED = 42
EPOCHS = 100
BATCH_SIZE = 128
NOISE_DIM = 100
LATENT_DIM = 16
BETA = 2.0
FID_SAMPLES = 5_000

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Data preparation

In [ ]:
(x_train_raw, _), (x_test_raw, _) = tf.keras.datasets.mnist.load_data()

x_train_gan = x_train_raw[..., None].astype("float32")
x_train_gan = (x_train_gan - 127.5) / 127.5

x_train_vae = x_train_raw[..., None].astype("float32") / 255.0
x_test_vae = x_test_raw[..., None].astype("float32") / 255.0

gan_dataset = (
    tf.data.Dataset.from_tensor_slices(x_train_gan)
    .shuffle(len(x_train_gan), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for axis, image in zip(axes.flat, x_train_raw[:16]):
    axis.imshow(image, cmap="gray")
    axis.axis("off")
fig.suptitle("MNIST training samples")
plt.tight_layout()
plt.show()

print("Training images:", x_train_raw.shape)
print("Test images:", x_test_raw.shape)

## 3. DCGAN architecture and training

In [ ]:
def make_generator(noise_dim=NOISE_DIM):
    return tf.keras.Sequential([
        layers.Input(shape=(noise_dim,)),
        layers.Dense(7 * 7 * 128, use_bias=False),
        layers.BatchNormalization(momentum=0.8),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Reshape((7, 7, 128)),
        layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),
        layers.BatchNormalization(momentum=0.8),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),
        layers.BatchNormalization(momentum=0.8),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Conv2DTranspose(1, 5, padding="same", use_bias=False, activation="tanh"),
    ], name="dcgan_generator")


def make_discriminator():
    return tf.keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, strides=2, padding="same"),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.25),
        layers.Conv2D(64, 3, strides=2, padding="same"),
        layers.BatchNormalization(momentum=0.8),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.25),
        layers.Conv2D(128, 3, strides=2, padding="same"),
        layers.BatchNormalization(momentum=0.8),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.25),
        layers.Conv2D(256, 3, padding="same"),
        layers.BatchNormalization(momentum=0.8),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(1, activation="sigmoid"),
    ], name="dcgan_discriminator")


generator = make_generator()
discriminator = make_discriminator()
generator.summary()
discriminator.summary()

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)
generator_optimizer = tf.keras.optimizers.Adam(1e-3, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-3, beta_1=0.5)


@tf.function
def gan_train_step(real_images):
    current_batch_size = tf.shape(real_images)[0]
    noise = tf.random.normal([current_batch_size, NOISE_DIM])

    with tf.GradientTape() as generator_tape, tf.GradientTape() as discriminator_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(real_images, training=True)
        fake_output = discriminator(generated_images, training=True)

        generator_loss = bce(tf.ones_like(fake_output), fake_output)
        discriminator_loss = (
            bce(tf.ones_like(real_output), real_output)
            + bce(tf.zeros_like(fake_output), fake_output)
        )

    generator_gradients = generator_tape.gradient(
        generator_loss, generator.trainable_variables
    )
    discriminator_gradients = discriminator_tape.gradient(
        discriminator_loss, discriminator.trainable_variables
    )
    generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(
        zip(discriminator_gradients, discriminator.trainable_variables)
    )

    real_accuracy = tf.reduce_mean(tf.cast(real_output > 0.5, tf.float32))
    fake_accuracy = tf.reduce_mean(tf.cast(fake_output <= 0.5, tf.float32))
    return generator_loss, discriminator_loss, real_accuracy, fake_accuracy


def train_gan(dataset, epochs):
    history = {"generator_loss": [], "discriminator_loss": [], "real_accuracy": [], "fake_accuracy": []}
    start = time.time()
    for epoch in range(epochs):
        metrics = [tf.keras.metrics.Mean() for _ in range(4)]
        for batch in dataset:
            values = gan_train_step(batch)
            for metric, value in zip(metrics, values):
                metric.update_state(value)
        values = [float(metric.result()) for metric in metrics]
        for key, value in zip(history, values):
            history[key].append(value)
        print(
            f"Epoch {epoch + 1:03d}/{epochs} | "
            f"G {values[0]:.4f} | D {values[1]:.4f} | "
            f"real acc {values[2]:.3f} | fake acc {values[3]:.3f}"
        )
    print(f"GAN training time: {(time.time() - start) / 60:.1f} minutes")
    return history


gan_history = train_gan(gan_dataset, EPOCHS)

## 4. Convolutional beta-VAE architecture and training

In [ ]:
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon


def make_encoder(latent_dim=LATENT_DIM):
    inputs = layers.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(inputs)
    x = layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    z_mean = layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
    z = Sampling()([z_mean, z_log_var])
    return Model(inputs, [z_mean, z_log_var, z], name="vae_encoder")


def make_decoder(latent_dim=LATENT_DIM):
    return tf.keras.Sequential([
        layers.Input(shape=(latent_dim,)),
        layers.Dense(7 * 7 * 128, activation="relu"),
        layers.Reshape((7, 7, 128)),
        layers.Conv2DTranspose(128, 3, activation="relu", padding="same"),
        layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same"),
        layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same"),
        layers.Conv2DTranspose(1, 3, activation="sigmoid", padding="same"),
    ], name="vae_decoder")


class BetaVAE(Model):
    def __init__(self, encoder, decoder, beta=BETA):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta
        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]

    def call(self, inputs):
        _, _, z = self.encoder(inputs)
        return self.decoder(z)

    def compute_losses(self, data):
        z_mean, z_log_var, z = self.encoder(data)
        reconstruction = self.decoder(z)
        reconstruction_loss = tf.reduce_mean(
            tf.keras.losses.binary_crossentropy(data, reconstruction)
        ) * 28 * 28
        kl_loss = -0.5 * tf.reduce_mean(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
        )
        return reconstruction_loss + self.beta * kl_loss, reconstruction_loss, kl_loss

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]
        with tf.GradientTape() as tape:
            losses = self.compute_losses(data)
        gradients = tape.gradient(losses[0], self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))
        for metric, value in zip(self.metrics, losses):
            metric.update_state(value)
        return {metric.name: metric.result() for metric in self.metrics}

    def test_step(self, data):
        if isinstance(data, tuple):
            data = data[0]
        losses = self.compute_losses(data)
        for metric, value in zip(self.metrics, losses):
            metric.update_state(value)
        return {metric.name: metric.result() for metric in self.metrics}


encoder = make_encoder()
decoder = make_decoder()
vae = BetaVAE(encoder, decoder)
vae_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3, beta_1=0.5)
vae.compile(optimizer=vae_optimizer)
vae_history = vae.fit(
    x_train_vae,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    shuffle=True,
)

## 5. Training diagnostics and generated samples

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(gan_history["generator_loss"], label="Generator")
axes[0, 0].plot(gan_history["discriminator_loss"], label="Discriminator")
axes[0, 0].set_title("DCGAN losses")
axes[0, 0].legend()

axes[0, 1].plot(gan_history["real_accuracy"], label="Real")
axes[0, 1].plot(gan_history["fake_accuracy"], label="Fake")
axes[0, 1].set_title("Discriminator accuracy")
axes[0, 1].legend()

axes[1, 0].plot(vae_history.history["loss"], label="Train")
axes[1, 0].plot(vae_history.history["val_loss"], label="Validation")
axes[1, 0].set_title("Beta-VAE total loss")
axes[1, 0].legend()

axes[1, 1].plot(vae_history.history["kl_loss"], label="Train KL")
axes[1, 1].plot(vae_history.history["val_kl_loss"], label="Validation KL")
axes[1, 1].set_title("Beta-VAE KL loss")
axes[1, 1].legend()

for axis in axes.flat:
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

sample_seed = tf.random.normal([16, NOISE_DIM], seed=SEED)
gan_samples = (generator(sample_seed, training=False) + 1.0) / 2.0
vae_latents = tf.random.normal([16, LATENT_DIM], seed=SEED)
vae_samples = decoder(vae_latents, training=False)

fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for i in range(16):
    axes.flat[i].imshow(gan_samples[i, ..., 0], cmap="gray")
    axes.flat[i].axis("off")
    axes.flat[i + 16].imshow(vae_samples[i, ..., 0], cmap="gray")
    axes.flat[i + 16].axis("off")
axes.flat[0].set_title("DCGAN", loc="left")
axes.flat[16].set_title("Beta-VAE", loc="left")
plt.tight_layout()
plt.show()

## 6. Shared feature-space evaluation

Both models are compared against the same deterministic sample from the MNIST test set. The score uses InceptionV3 features for continuity with the original experiment, but it is reported as an internal comparison because MNIST differs substantially from ImageNet.

In [ ]:
def extract_features(feature_model, images, batch_size=64):
    dataset = tf.data.Dataset.from_tensor_slices(images).batch(batch_size)
    feature_batches = []
    for batch in dataset:
        batch = tf.cast(batch, tf.float32)
        if batch.shape.rank == 3:
            batch = batch[..., None]
        batch = tf.image.grayscale_to_rgb(batch)
        batch = tf.image.resize(batch, (299, 299))
        batch = preprocess_input(batch)
        feature_batches.append(feature_model(batch, training=False).numpy())
    return np.concatenate(feature_batches, axis=0)


def feature_distance(feature_model, real_images, generated_images):
    real_features = extract_features(feature_model, real_images)
    generated_features = extract_features(feature_model, generated_images)
    real_mean, real_covariance = real_features.mean(0), np.cov(real_features, rowvar=False)
    generated_mean = generated_features.mean(0)
    generated_covariance = np.cov(generated_features, rowvar=False)
    covariance_mean = sqrtm(real_covariance @ generated_covariance)
    if np.iscomplexobj(covariance_mean):
        covariance_mean = covariance_mean.real
    return float(
        np.sum((real_mean - generated_mean) ** 2)
        + np.trace(real_covariance + generated_covariance - 2 * covariance_mean)
    )


rng = np.random.default_rng(SEED)
reference_indices = rng.choice(len(x_test_raw), size=FID_SAMPLES, replace=False)
real_reference = x_test_raw[reference_indices].astype("float32")

gan_eval_noise = tf.random.normal([FID_SAMPLES, NOISE_DIM], seed=SEED)
gan_generated = (generator(gan_eval_noise, training=False) + 1.0) * 127.5

vae_eval_latents = tf.random.normal([FID_SAMPLES, LATENT_DIM], seed=SEED)
vae_generated = decoder(vae_eval_latents, training=False) * 255.0

inception = InceptionV3(include_top=False, pooling="avg", input_shape=(299, 299, 3))
gan_score = feature_distance(inception, real_reference, gan_generated)
vae_score = feature_distance(inception, real_reference, vae_generated)

print(f"DCGAN comparative feature distance: {gan_score:.4f}")
print(f"Beta-VAE comparative feature distance: {vae_score:.4f}")
print("Lower is better within this shared experimental setup.")

## 7. Interpretation checklist

After running the notebook, compare:

- The clarity and diversity of samples from each model
- Whether adversarial losses remain bounded without visible mode collapse
- The gap between beta-VAE training and validation losses
- Reconstruction quality alongside random latent samples
- The shared feature-distance scores, interpreted with the MNIST/ImageNet limitation above

Stronger follow-up experiments should repeat training with several random seeds and add a digit-classifier embedding metric or KID.